# 模型持久化、导出与一致性验证

## 学习目标

能够区分 state_dict、训练检查点和 TorchScript，并验证保存前后的输出一致。


## 概念模型与执行路径

state_dict 依赖 Python 模型定义，训练检查点还包含优化状态，TorchScript 保存可独立加载的执行图。导出成功只是格式正确，仍需用代表性输入验证数值和动态行为。


### 实验 1：定位模型与导出入口

**实验目的**：定位课程根目录，为导入 `ImageClassifier` 和运行导出脚本做准备。该代码只解决 notebook 启动目录差异；找不到 `common` 时应先检查工作目录。


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：将 eager 模型 trace 为 TorchScript

**实验目的**：以 `(2,1,28,28)` 示例输入记录 `ImageClassifier` 的张量运算路径。导出前调用 `eval()` 固定模式，并用 `inference_mode()` 避免构图。

`torch.jit.trace` 通过实际执行捕获算子，不理解任意 Python 控制流；适合当前数据无关前向。`eager_output` 是一致性基线。trace 成功只说明示例路径可捕获，不代表所有输入都正确。


In [ ]:
import tempfile
import torch
from common.models import ImageClassifier
model = ImageClassifier().eval()
example = torch.randn(2, 1, 28, 28)
with torch.inference_mode():
    eager_output = model(example)
    scripted = torch.jit.trace(model, example)
print(eager_output.shape)


### 实验 3：保存、加载并验证数值一致性

**实验目的**：保存 TorchScript artifact 后独立加载，并用 `assert_close` 验证相同输入上的输出与 eager 基线一致。临时目录退出后自动删除。

加载后仍显式 `eval()`。round-trip 测试覆盖序列化格式、参数与执行图，而不仅是文件是否存在。


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    destination = Path(directory) / "model.pt"
    scripted.save(str(destination))
    loaded = torch.jit.load(str(destination)).eval()
    with torch.inference_mode():
        loaded_output = loaded(example)
    torch.testing.assert_close(eager_output, loaded_output)
    print("round-trip bytes:", destination.stat().st_size)


### 实验 4：验证 batch 维的动态性

**实验目的**：用 batch size 5 调用由 batch size 2 trace 的模型，确认 batch 维未被固化，输出应为 `(5,10)`。这不自动证明空间尺寸或所有边界输入均受支持。


In [ ]:
different_batch = torch.randn(5, 1, 28, 28)
with torch.inference_mode():
    print("different batch shape:", loaded(different_batch).shape)


### 实验 5：运行完整导出脚本

**实验目的**：在终端执行课程导出入口，生成 artifact 并运行脚本内一致性检查。`--quick` 用于轻量冒烟；产物仍需在目标运行时、版本和设备上验证。


In [ ]:
# python 07-deep-learning/pytorch/examples/export_model.py --quick


## 底层机制

Tracing 记录示例输入经过的运算路径，数据依赖的 Python 控制流可能被固化。导出验证应覆盖不同 batch 和边界输入。生产部署还要固定预处理、类别映射和版本。


## 检查点

为什么只比较文件是否生成不足以证明导出正确？state_dict 与 TorchScript 各自依赖什么？


## 试一试

导出后用 batch size 1 和 5 验证输出；再加入依赖输入值的 Python if，观察 tracing 警告或行为差异。


## 常见错误与调试

导出时未 eval、遗漏预处理规范、只测试一个输入、把训练检查点当成可直接部署模型。
